# Intent labeling — MongoDB + Gemini 2.5 Pro

Pipeline độc lập để kiểm định nhãn GPT-4o-mini:

```text
data/hasaki_prelabel.json
→ retrieve top-k candidate intent từ MongoDB
→ Gemini 2.5 Pro gán nhãn JSON
→ validate taxonomy + guardrail
→ export data/raw/hasaki/hasaki_labelled_gemini_2_5_pro.json
```

**Lưu ý:** Gemini **không** được xem nhãn GPT trong prompt. File `hasaki_labelled_clean.json` chỉ dùng ở bước so sánh cuối.

Xem plan: `docs/gemini_2_5_pro_labeling_eval_plan.md`


## 1. Cài đặt

> Chạy trên Colab hoặc local. Cần `GEMINI_API_KEY`, `MONGODB_URI`, và graph đã có trong MongoDB (`intent_nodes` / `intent_edges`).


In [ ]:
import sys
print(sys.executable)


In [ ]:
pip install -q pymongo certifi sentence-transformers scikit-learn numpy google-genai


## 1.5. Mount Google Drive (Colab)

Khi chạy trên Colab, cell này mount Drive và set `INTENT_REPO`.


In [ ]:
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
DEFAULT_DRIVE_REPO = "/content/drive/MyDrive/intent_kb"

if IN_COLAB:
    if not Path("/content/drive/MyDrive").exists():
        from google.colab import drive
        drive.mount("/content/drive")

    os.environ.setdefault("INTENT_REPO", DEFAULT_DRIVE_REPO)
    repo = Path(os.environ["INTENT_REPO"])
    print(f"INTENT_REPO = {repo}")
    expected = repo / "data" / "hasaki_prelabel.json"
    print(f"hasaki_prelabel.json: {'OK' if expected.exists() else 'MISSING'} {expected}")
else:
    print("Khong phai Colab — INTENT_REPO lay tu env hoac thu muc repo hien tai.")


## 2. Cấu hình


In [ ]:
import os
import getpass
from pathlib import Path

REPO_ROOT = Path(os.environ.get("INTENT_REPO", ".")).resolve()
HASAKI_JSON = REPO_ROOT / "data/hasaki_prelabel.json"
GPT_LABEL_FILE = REPO_ROOT / "data/raw/hasaki/hasaki_labelled_clean.json"

_out_env = (os.environ.get("INTENT_OUTPUT_DIR") or "").strip()
OUTPUT_DIR = Path(_out_env).resolve() if _out_env else (REPO_ROOT / "data" / "outputs" / "gemini")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

GEMINI_OUTPUT_FILE = REPO_ROOT / "data/raw/hasaki/hasaki_labelled_gemini_2_5_pro.json"
GEMINI_OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)

DB_NAME = os.environ.get("MONGODB_DB", "intent_kb")
COL_NODES = "intent_nodes"
COL_EDGES = "intent_edges"

MONGODB_URI = (os.environ.get("MONGODB_URI") or "").strip()
if not MONGODB_URI:
    MONGODB_URI = getpass.getpass("MONGODB_URI: ").strip()

GEMINI_MODEL = os.environ.get("GEMINI_MODEL", "gemini-2.5-pro")
GEMINI_TEMPERATURE = float(os.environ.get("GEMINI_TEMPERATURE", "0"))
GEMINI_MAX_RETRIES = int(os.environ.get("GEMINI_MAX_RETRIES", "3"))
GEMINI_MAX_OUTPUT_TOKENS = int(os.environ.get("GEMINI_MAX_OUTPUT_TOKENS", "512"))
GEMINI_API_KEY = (os.environ.get("GEMINI_API_KEY") or "").strip()

TOP_K_CANDIDATES = int(os.environ.get("TOP_K_CANDIDATES", "12"))
MIN_CONF_APPROVE = float(os.environ.get("MIN_CONF_APPROVE", "0.80"))

HASAKI_BATCH_SIZE = int(os.environ.get("HASAKI_BATCH_SIZE", "50"))
HASAKI_RUN_OFFSET = int(os.environ.get("HASAKI_RUN_OFFSET", "0"))
DRY_RUN_LIMIT = int(os.environ.get("DRY_RUN_LIMIT", "0"))  # >0: chi chay N mau dau (test prompt)

print("HASAKI_JSON:", HASAKI_JSON.exists(), HASAKI_JSON)
print("GPT_LABEL_FILE:", GPT_LABEL_FILE.exists(), GPT_LABEL_FILE)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("GEMINI_OUTPUT_FILE:", GEMINI_OUTPUT_FILE)
print("GEMINI_MODEL:", GEMINI_MODEL)
print("MONGODB_URI set:", bool(MONGODB_URI))


## 2.5. Nhập GEMINI_API_KEY


In [ ]:
if not GEMINI_API_KEY:
    GEMINI_API_KEY = getpass.getpass("GEMINI_API_KEY: ").strip()

if not GEMINI_API_KEY:
    raise ValueError("GEMINI_API_KEY rong — set env hoac nhap o cell nay.")

os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY
print(f"GEMINI_API_KEY set (len={len(GEMINI_API_KEY)}, suffix=...{GEMINI_API_KEY[-4:]})")


## 3. Kết nối MongoDB


In [ ]:
import certifi
from pymongo import MongoClient

client = MongoClient(
    MONGODB_URI,
    tls=True,
    tlsCAFile=certifi.where(),
    serverSelectionTimeoutMS=15000,
)
client.admin.command("ping")
db = client[DB_NAME]
print("OK db:", db.name)
print("intent_nodes:", db[COL_NODES].count_documents({}))
print("intent_edges:", db[COL_EDGES].count_documents({}))


## 4. Helper: retrieval + guardrail

Logic giữ nguyên như `intent_labeling_mongodb_qwen.ipynb` để candidate intent và validate taxonomy nhất quán giữa GPT và Gemini.


In [ ]:
from datetime import datetime, timezone
import json
import re
import unicodedata
import time
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

SEMANTIC_SNAP_TO_INTENT_THRESHOLD = 0.85
_semantic_model = None

def get_semantic_model():
    global _semantic_model
    if _semantic_model is None:
        print("Loading paraphrase-multilingual-MiniLM-L12-v2...")
        _semantic_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
    return _semantic_model

def build_graph_aware_intent_text(intent_doc, db):
    l1, l2, l3 = intent_doc.get("l1", ""), intent_doc.get("l2", ""), intent_doc.get("l3", "")
    description = intent_doc.get("description", "")
    detection_signals = intent_doc.get("detection_signals", "")
    example = intent_doc.get("example", "")
    full_path = f"{l1}.{l2}.{l3}"
    siblings_cursor = db[COL_NODES].find(
        {"level": "intent", "l2": l2, "_id": {"$ne": intent_doc["_id"]}}, {"l3": 1}
    ).limit(5)
    siblings = [doc.get("l3", "") for doc in siblings_cursor if doc.get("l3")]
    siblings_text = ", ".join(siblings) if siblings else "None"
    return f"""Intent: {full_path}.
Parent: {l2}.
Siblings: {siblings_text}.
Description: {description}
Signals: {detection_signals}
Example: {example}"""

def embed_and_store_intent_nodes(db, *, force=False):
    model = get_semantic_model()
    col_nodes = db[COL_NODES]
    intent_nodes = list(col_nodes.find({"level": "intent"}))
    updated_count = skipped_count = 0
    for intent_doc in intent_nodes:
        if not force and "embedding" in intent_doc and "graph_aware_text" in intent_doc:
            skipped_count += 1
            continue
        graph_text = build_graph_aware_intent_text(intent_doc, db)
        embedding = model.encode([graph_text])[0]
        embedding = embedding / np.linalg.norm(embedding)
        col_nodes.update_one(
            {"_id": intent_doc["_id"]},
            {"$set": {
                "graph_aware_text": graph_text,
                "embedding": embedding.tolist(),
                "embedding_model": "paraphrase-multilingual-MiniLM-L12-v2",
                "embedding_updated_at": datetime.now(timezone.utc),
            }},
        )
        updated_count += 1
    print(f"Embedded {updated_count} | Skipped {skipped_count} | Total {len(intent_nodes)}")
    return updated_count

def semantic_retrieval(db, query_text, top_k=10, min_similarity=0.3):
    model = get_semantic_model()
    query_embedding = model.encode([query_text])[0]
    query_embedding = query_embedding / np.linalg.norm(query_embedding)
    intent_nodes = list(db[COL_NODES].find(
        {"level": "intent", "embedding": {"$exists": True}},
        {"_id": 1, "name": 1, "l1": 1, "l2": 1, "l3": 1, "description": 1,
         "detection_signals": 1, "example": 1, "product_category": 1, "embedding": 1},
    ))
    scored = []
    for intent_doc in intent_nodes:
        stored_embedding = np.array(intent_doc["embedding"])
        similarity = cosine_similarity([query_embedding], [stored_embedding])[0][0]
        if similarity >= min_similarity:
            intent_doc["semantic_score"] = float(similarity)
            scored.append(intent_doc)
    scored.sort(key=lambda x: x["semantic_score"], reverse=True)
    return scored[:top_k]

_HASAKI_BEAUTY_CATEGORIES_CACHE = None

def get_hasaki_beauty_categories():
    global _HASAKI_BEAUTY_CATEGORIES_CACHE
    path = HASAKI_JSON
    mtime = path.stat().st_mtime_ns if path.exists() else -1
    if isinstance(_HASAKI_BEAUTY_CATEGORIES_CACHE, tuple):
        cached, mt = _HASAKI_BEAUTY_CATEGORIES_CACHE
        if mt == mtime:
            return cached
    cats = set()
    if path.exists():
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        for row in data if isinstance(data, list) else []:
            c = (row.get("category") or "").strip()
            if c:
                cats.add(c)
    if not cats:
        cats = {"Sức khỏe làm đẹp", "Trang điểm", "Mỹ phẩm high end"}
    fs = frozenset(cats)
    _HASAKI_BEAUTY_CATEGORIES_CACHE = (fs, mtime)
    return fs

def infer_domain(sample):
    cat_raw = (sample.get("category") or "").strip()
    cat = cat_raw.lower()
    name = (sample.get("product_name") or "").lower()
    blob = f"{cat} {name}"
    if cat_raw and cat_raw in get_hasaki_beauty_categories():
        return "My pham"
    electronics_kw = "laptop phone dien thoai tai nghe chuot ban phim camera smartwatch gaming console ram gpu cpu may anh dong ho thong minh tablet iphone tivi tv"
    if any(k in blob for k in electronics_kw.split()):
        return "Dien tu"
    cat_hints = ("may tinh", "dien tu", "phu kien dien", "dien thoai", "dong ho thong minh")
    if any(h in cat for h in cat_hints):
        return "Dien tu"
    return "My pham"

def _intent_slug_blob(d):
    parts = [d.get("l1"), d.get("l2"), d.get("l3"), d.get("name")]
    return " ".join(str(p or "") for p in parts).lower()

_ELECTRONICS_SLUG_MARKERS = ("dien_thoai", "laptop", "may_anh", "smartwatch", "tai_nghe", "iphone", "macbook", "gpu", "chuot")

def _candidate_looks_electronics_intent(d):
    s = _intent_slug_blob(d)
    return any(m in s for m in _ELECTRONICS_SLUG_MARKERS)

def _sentence_mentions_electronics(text):
    t = (text or "").lower()
    hints = ("dien thoai", "dien thoai", "laptop", "iphone", "macbook", "may anh", "máy ảnh", "tai nghe", "smartwatch")
    return any(h in t for h in hints)

def _category_match_bonus(sample, d):
    if not sample:
        return 0
    cat = (sample.get("category") or "").lower()
    pc = (d.get("product_category") or "").lower()
    if not cat or not pc:
        return 0
    cat_tokens = [w for w in re.findall(r"\w+", cat, flags=re.UNICODE) if len(w) >= 3]
    return min(1, sum(1 for w in cat_tokens if w in pc))

def get_candidate_l3_from_mongodb(db, text, category=None, top_k=10, sample=None):
    tokens = [t.lower() for t in re.findall(r"\w+", text or "", flags=re.UNICODE) if len(t) >= 2]
    if not tokens:
        return []
    or_conditions = []
    for tok in tokens:
        rgx = {"$regex": re.escape(tok), "$options": "i"}
        or_conditions.extend([
            {"name": rgx}, {"description": rgx}, {"detection_signals": rgx},
            {"example": rgx}, {"l3": rgx},
        ])
    docs = list(db[COL_NODES].find(
        {"level": "intent", "$or": or_conditions},
        {"_id": 1, "name": 1, "l1": 1, "l2": 1, "l3": 1, "description": 1,
         "detection_signals": 1, "example": 1, "product_category": 1},
    ))
    beauty_ctx = bool(sample) and infer_domain(sample) == "My pham"
    scored = []
    token_set = set(tokens)
    for d in docs:
        if beauty_ctx and _candidate_looks_electronics_intent(d) and not _sentence_mentions_electronics(text):
            continue
        blob = " ".join([
            str(d.get("name", "")), str(d.get("description", "")),
            str(d.get("detection_signals", "")), str(d.get("example", "")),
            str(d.get("l3", "")), str(d.get("product_category", "")),
        ]).lower()
        hit = sum(1 for t in token_set if t in blob)
        if hit <= 0:
            continue
        cat_bonus = _category_match_bonus(sample, d) if sample else 0
        scored.append((hit + 0.2 * cat_bonus, d))
    scored.sort(key=lambda x: x[0], reverse=True)
    pool = [d for _, d in scored[: max(top_k * 4, top_k + 8)]]
    seen, uniq = set(), []
    for d in pool:
        key = (d.get("l1"), d.get("l2"), d.get("l3"))
        if key in seen:
            continue
        seen.add(key)
        uniq.append(d)
        if len(uniq) >= top_k:
            break
    return uniq

def union_retrieval(db, text, category=None, top_k_regex=8, top_k_semantic=6, sample=None):
    regex_candidates = get_candidate_l3_from_mongodb(db, text, category, top_k_regex, sample)
    semantic_candidates = semantic_retrieval(db, text, top_k_semantic)
    seen_ids, union_candidates = set(), []
    for candidate in regex_candidates:
        if candidate["_id"] not in seen_ids:
            candidate["retrieval_method"] = "regex"
            union_candidates.append(candidate)
            seen_ids.add(candidate["_id"])
    for candidate in semantic_candidates:
        if candidate["_id"] not in seen_ids:
            candidate["retrieval_method"] = "semantic"
            union_candidates.append(candidate)
            seen_ids.add(candidate["_id"])
    max_total = max(top_k_regex + top_k_semantic, TOP_K_CANDIDATES)
    return union_candidates[:max_total]

def _norm_label(s):
    s = unicodedata.normalize("NFC", (s or "").strip())
    return re.sub(r"\s+", " ", s).lower()

def _collect_allowed_taxonomy(db):
    l1_set, l2_set, l3_set = set(), set(), set()
    for doc in db[COL_NODES].find({"level": "l1"}, {"name": 1}):
        if doc.get("name"):
            l1_set.add(doc["name"])
    for doc in db[COL_NODES].find({"level": "l2"}, {"name": 1}):
        if doc.get("name"):
            l2_set.add(doc["name"])
    for doc in db[COL_NODES].find({"level": "l3"}, {"name": 1}):
        if doc.get("name"):
            l3_set.add(doc["name"])

    def _norm_to_canon(name_set):
        m = {}
        for name in sorted(name_set):
            k = _norm_label(name)
            if k and k not in m:
                m[k] = name
        return m

    return {
        "l1": l1_set, "l2": l2_set, "l3": l3_set,
        "norm_canon_l1": _norm_to_canon(l1_set),
        "norm_canon_l2": _norm_to_canon(l2_set),
        "norm_canon_l3": _norm_to_canon(l3_set),
    }

def _snap_pred_to_canonical_taxonomy(pred, allowed):
    for lvl, key in [("l1", "L1"), ("l2", "L2"), ("l3", "L3")]:
        raw = pred.get(key)
        if raw is None:
            continue
        raw = str(raw).strip()
        if not raw:
            continue
        canon = allowed.get(f"norm_canon_{lvl}", {}).get(_norm_label(raw))
        if canon:
            pred[key] = canon

def _is_label_allowed(pred, allowed):
    n1, n2, n3 = _norm_label(pred.get("L1")), _norm_label(pred.get("L2")), _norm_label(pred.get("L3"))
    nl1 = {_norm_label(x) for x in allowed["l1"]}
    nl2 = {_norm_label(x) for x in allowed["l2"]}
    nl3 = {_norm_label(x) for x in allowed["l3"]}
    return (n1 in nl1) and (n2 in nl2) and (n3 in nl3)

_ALLOWED_L1_NORM_TO_CANON = {
    "truoc_mua_hang": "truoc_mua_hang", "truoc mua hang": "truoc_mua_hang",
    "truoc_ban": "truoc_mua_hang", "truoc ban": "truoc_mua_hang",
    "trước mua hàng": "truoc_mua_hang", "trước bán": "truoc_mua_hang", "trước ban": "truoc_mua_hang",
    "truoc bán": "truoc_mua_hang",
    "sau_mua_hang": "sau_mua_hang", "sau mua hang": "sau_mua_hang",
    "sau_ban": "sau_mua_hang", "sau ban": "sau_mua_hang",
    "sau mua hàng": "sau_mua_hang", "sau bán": "sau_mua_hang",
}
_L2_SLUGS_OFTEN_CONFUSED_AS_L1 = {
    "gia_so_sanh", "san_pham_so_sanh", "thiet_bi_tu_van", "chat_luong_loi",
    "di_ung_an_toan", "ton_kho_kiem_tra", "uu_dai_yeu_cau", "khieu_nai_dich_vu",
    "trang_thai_van_chuyen", "thay_doi_don_hang", "doi_tra", "hoan_tien_yeu_cau",
    "hong_vo_san_pham", "giao_hang_loi", "van_de_giao_hang", "thong_tin_san_pham",
}
_SLUG_RE = re.compile(r"^[a-z][a-z0-9_]+$")

def _validate_pred_for_auto_create(pred):
    out = {"L1": "", "L2": "", "L3": ""}
    l1_raw, l2_raw, l3_raw = (pred.get("L1") or "").strip(), (pred.get("L2") or "").strip(), (pred.get("L3") or "").strip()
    if not (l1_raw and l2_raw and l3_raw):
        return False, "empty_l1_l2_l3", out
    l1_canon = _ALLOWED_L1_NORM_TO_CANON.get(_norm_label(l1_raw))
    if not l1_canon:
        return False, f"l1_must_be_truoc_or_sau_mua_hang_got:{l1_raw!r}", out
    out["L1"] = l1_canon
    if not _SLUG_RE.match(l2_raw):
        return False, f"l2_invalid_slug:{l2_raw!r}", out
    out["L2"] = l2_raw
    if not _SLUG_RE.match(l3_raw):
        return False, f"l3_invalid_slug:{l3_raw!r}", out
    if l3_raw == l2_raw:
        return False, "l3_eq_l2", out
    out["L3"] = l3_raw
    return True, "", out

def is_sentence_too_ambiguous_to_annotate(sentence, min_length=6, min_meaningful_words=1):
    if not sentence or not sentence.strip():
        return True, "empty_sentence"
    text = sentence.strip()
    char_count = len(re.sub(r"\s+", "", text))
    if char_count < min_length:
        return True, f"too_short_chars_{char_count}"
    words = re.findall(r"\w+", text.lower(), flags=re.UNICODE)
    stop_words = {"có", "thế", "này", "kia", "được", "như"}
    meaningful_words = [w for w in words if len(w) >= 2 and w not in stop_words]
    money_words = {"tiền", "giá", "bao", "nhiều", "cost", "price"}
    product_words = {"sản", "phẩm", "sp", "hàng", "tốt", "xấu", "quality"}
    if any(w in " ".join(meaningful_words) for w in money_words + product_words):
        min_meaningful_words = 1
    if len(meaningful_words) < min_meaningful_words:
        return True, f"too_few_meaningful_words_{len(meaningful_words)}"
    return False, ""

def build_retrieval_prompt(sample, candidates):
    vertical = infer_domain(sample)
    lines = []
    for i, c in enumerate(candidates, 1):
        pc = c.get("product_category") or ""
        lines.append(
            f"{i}. L1={c.get('l1')} | L2={c.get('l2')} | L3={c.get('l3')} | product_category={pc!r} | intent_id={c.get('_id')}"
        )
    taxonomy_text = "\n".join(lines) if lines else (
        "KHONG CO UNG VIEN nao tu MongoDB. Ban van PHAI de xuat bo (L1, L2, L3) hop le theo quy tac ben duoi va dat is_new_label=true."
    )
    parts = [
        "Bạn là chuyên gia gán nhãn intent cho QA thương mại điện tử (mỹ phẩm / làm đẹp và điện tử).",
        "Nhiệm vụ: chọn intent dựa trên NGỮ NGHĨA câu hỏi của khách, KHÔNG dựa vào category/tên sản phẩm.",
        f"Ngành ước lượng (chỉ dùng để loại trừ slug ngành sai, KHÔNG dùng để chọn intent): **{vertical}**.",
        "",
        "QUY TẮC TAXONOMY (BẮT BUỘC):",
        "- L1 ∈ {'truoc_mua_hang', 'sau_mua_hang'} — viết đúng underscore + lowercase ASCII.",
        "- L2, L3 là slug ASCII lowercase + underscore.",
        "- L3 phải KHÁC L2.",
        "",
        "QUY TẮC CHỌN NHÃN:",
        "- sentence là nguồn quyết định chính.",
        "- category, product_name, brand chỉ là context phụ.",
        "- Nếu CÓ ứng viên phù hợp: copy CHÍNH XÁC L1/L2/L3, đặt is_new_label=false.",
        "- Nếu KHÔNG có ứng viên phù hợp: đề xuất nhãn mới, đặt is_new_label=true.",
        "- Confidence ∈ [0,1].",
        "",
        "ỨNG VIÊN (MongoDB):",
        taxonomy_text,
        "",
        "NGỮ CẢNH ĐẦU VÀO:",
        f"- sample_id: {sample.get('sample_id', '')}",
        f"- Câu (sentence) [CHÍNH]: {sample.get('sentence', '')}",
        f"- Danh mục (category) [phụ]: {sample.get('category', '')}",
        f"- Tên sản phẩm (product_name) [phụ]: {sample.get('product_name', '')}",
        f"- Thương hiệu (brand) [phụ]: {sample.get('brand', '')}",
        "",
        "Trả về DUY NHẤT một đối tượng JSON hợp lệ (KHÔNG markdown):",
        "{",
        '  "reasoning": "...",',
        '  "confidence": 0.0,',
        '  "L1": "...",',
        '  "L2": "...",',
        '  "L3": "...",',
        '  "is_new_label": false',
        "}",
    ]
    return "\n".join(parts)

def serialize_candidates(cands):
    out = []
    for c in cands or []:
        out.append({
            "level_1": c.get("l1") or "",
            "level_2": c.get("l2") or "",
            "level_3": c.get("l3") or "",
            "intent_id": c.get("_id") or "",
            "retrieval_method": c.get("retrieval_method"),
            "semantic_score": c.get("semantic_score"),
        })
    return out

def enrich_sample(s):
    out = dict(s)
    out["domain"] = infer_domain(s)
    return out

def evaluate_gemini_prediction(pred):
    allowed = _collect_allowed_taxonomy(db)
    pred = dict(pred or {})
    _snap_pred_to_canonical_taxonomy(pred, allowed)
    confidence = float(pred.get("confidence") or 0)
    is_new = bool(pred.get("is_new_label"))
    valid_existing = _is_label_allowed(pred, allowed)

    if pred.get("_ambiguous"):
        return pred, "skipped_ambiguous", confidence

    if valid_existing:
        qa_status = "approved" if confidence >= MIN_CONF_APPROVE else "needs_review"
        return pred, qa_status, confidence

    if is_new:
        ok, reason, _norm = _validate_pred_for_auto_create(pred)
        if ok:
            return pred, "pending_new_label_review", confidence
        return pred, "pending_new_label_invalid_slug", confidence

    return pred, "rejected", confidence

print("Helpers OK")


## 5. Gemini client + gán nhãn


In [ ]:
from google import genai
from google.genai import types

gemini_client = genai.Client(api_key=GEMINI_API_KEY)
print(f"Gemini client OK — model: {GEMINI_MODEL}")

SYSTEM_PROMPT = (
    "Bạn chỉ trả về một đoạn JSON hợp lệ, không markdown, không text ngoài JSON. "
    "Tuân thủ thứ tự khóa: reasoning trước, sau đó confidence, L1, L2, L3, is_new_label."
)


def extract_json_object(text):
    text = (text or "").strip()
    text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\s*```\s*$", "", text)
    start, end = text.find("{"), text.rfind("}")
    if start == -1 or end == -1 or end <= start:
        raise ValueError("Khong tim thay JSON")
    return json.loads(text[start : end + 1])


def call_gemini(user_prompt):
    last_err = None
    for attempt in range(GEMINI_MAX_RETRIES):
        try:
            resp = gemini_client.models.generate_content(
                model=GEMINI_MODEL,
                contents=[
                    types.Content(
                        role="user",
                        parts=[types.Part(text=f"{SYSTEM_PROMPT}\n\n{user_prompt}")],
                    )
                ],
                config=types.GenerateContentConfig(
                    temperature=GEMINI_TEMPERATURE,
                    top_p=1,
                    max_output_tokens=GEMINI_MAX_OUTPUT_TOKENS,
                    response_mime_type="application/json",
                ),
            )
            return (resp.text or "").strip()
        except Exception as e:
            last_err = e
            time.sleep(min(2 ** attempt, 8))
    raise RuntimeError(f"Gemini call that bai sau {GEMINI_MAX_RETRIES} lan: {last_err}")


def predict_intent_gemini(sample, top_k=None, use_semantic=True):
    if top_k is None:
        top_k = TOP_K_CANDIDATES

    sentence = (sample.get("sentence") or "").strip()
    is_ambiguous, ambiguous_reason = is_sentence_too_ambiguous_to_annotate(sentence)
    if is_ambiguous:
        pred = {
            "reasoning": f"Cau qua ngan/mo ho de gan nhan: {ambiguous_reason}",
            "confidence": 0.0,
            "L1": "",
            "L2": "",
            "L3": "",
            "is_new_label": False,
            "_ambiguous": True,
            "_ambiguous_reason": ambiguous_reason,
        }
        return pred, f"SKIPPED_AMBIGUOUS: {ambiguous_reason}", []

    if use_semantic:
        cands = union_retrieval(
            db,
            sentence,
            category=sample.get("category"),
            top_k_regex=max(6, top_k // 2),
            top_k_semantic=max(4, top_k // 3),
            sample=sample,
        )
    else:
        cands = get_candidate_l3_from_mongodb(
            db, sentence, category=sample.get("category"), top_k=top_k, sample=sample
        )

    user_prompt = build_retrieval_prompt(sample, cands)
    raw = call_gemini(user_prompt)
    pred = extract_json_object(raw)
    return pred, raw, cands


def label_one_sample(sample):
    ex = enrich_sample(sample)
    try:
        pred, raw, cands = predict_intent_gemini(ex)
        pred, qa_status, confidence = evaluate_gemini_prediction(pred)
        return {
            "sample_id": ex.get("sample_id"),
            "product_id": ex.get("product_id"),
            "product_name": ex.get("product_name"),
            "brand": ex.get("brand"),
            "sentence": ex.get("sentence"),
            "category": ex.get("category"),
            "candidate_intents": serialize_candidates(cands),
            "gemini_intent_3_level": {
                "level_1": pred.get("L1") or "",
                "level_2": pred.get("L2") or "",
                "level_3": pred.get("L3") or "",
            },
            "gemini_confidence": confidence,
            "gemini_reasoning": pred.get("reasoning") or "",
            "gemini_is_new_label": bool(pred.get("is_new_label")),
            "gemini_qa_status": qa_status,
            "gemini_error": None,
            "raw_response": raw,
        }
    except Exception as e:
        return {
            "sample_id": ex.get("sample_id"),
            "product_id": ex.get("product_id"),
            "product_name": ex.get("product_name"),
            "brand": ex.get("brand"),
            "sentence": ex.get("sentence"),
            "category": ex.get("category"),
            "candidate_intents": [],
            "gemini_intent_3_level": {"level_1": "", "level_2": "", "level_3": ""},
            "gemini_confidence": 0.0,
            "gemini_reasoning": "",
            "gemini_is_new_label": False,
            "gemini_qa_status": "error",
            "gemini_error": str(e),
            "raw_response": None,
        }


## 6. Setup embeddings (chạy một lần nếu chưa có)

Bỏ qua nếu `intent_nodes` đã có field `embedding` từ notebook GPT/Qwen.


In [ ]:
FORCE_REEMBED = bool(int(os.environ.get("FORCE_REEMBED", "0")))
print(f"Embedding intent nodes (force={FORCE_REEMBED})...")
embed_count = embed_and_store_intent_nodes(db, force=FORCE_REEMBED)
print(f"Done. Newly embedded: {embed_count}")


## 7. Dry-run — thử 1 mẫu


In [ ]:
demo = enrich_sample({
    "sample_id": "demo_gemini",
    "product_id": "demo_001",
    "product_name": "Sua rua mat CeraVe",
    "brand": "CeraVe",
    "category": "Suc khoe lam dep",
    "sentence": "Shop oi gui don som cho minh dc ko",
    "source": "hasaki",
})
result = label_one_sample(demo)
print(json.dumps(result, ensure_ascii=False, indent=2))


## 8. Batch gán nhãn từ `hasaki_prelabel.json`

- Resume: set `HASAKI_RUN_OFFSET` ở §2.
- Dry-run nhanh: set `DRY_RUN_LIMIT=50` (hoặc 100) ở §2.
- Checkpoint sau mỗi batch vào `OUTPUT_DIR`.


In [ ]:
def run_gemini_batch(samples):
    stats = {}
    results = []
    for s in samples:
        row = label_one_sample(s)
        st = row.get("gemini_qa_status") or "unknown"
        stats[st] = stats.get(st, 0) + 1
        results.append(row)
        print(st, row.get("sample_id"))
    return {"stats": stats, "results": results}


def load_existing_gemini_results(path):
    if not path.exists():
        return {}
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    done = {}
    for row in data.get("results", []):
        sid = row.get("sample_id")
        if sid and not row.get("gemini_error"):
            done[sid] = row
    return done


if not HASAKI_JSON.exists():
    raise FileNotFoundError(f"Khong tim thay {HASAKI_JSON}")

with open(HASAKI_JSON, "r", encoding="utf-8") as f:
    hasaki_all = json.load(f)

n = len(hasaki_all)
off = int(HASAKI_RUN_OFFSET)
bs = int(HASAKI_BATCH_SIZE)
if off < 0 or off >= n:
    raise ValueError(f"HASAKI_RUN_OFFSET={off} khong hop le (tong {n})")

if DRY_RUN_LIMIT > 0:
    hasaki_all = hasaki_all[off : off + DRY_RUN_LIMIT]
    off = 0
    n = len(hasaki_all)
    print(f"DRY_RUN_LIMIT={DRY_RUN_LIMIT} -> chi chay {n} mau")

existing_done = load_existing_gemini_results(GEMINI_OUTPUT_FILE)
print(f"Resume: da co {len(existing_done)} mau trong {GEMINI_OUTPUT_FILE.name}")

agg_stats = {}
all_results = list(existing_done.values())
processed_ids = set(existing_done.keys())

for start in range(off, n, bs):
    chunk = hasaki_all[start : start + bs]
    pending = [s for s in chunk if s.get("sample_id") not in processed_ids]
    if not pending:
        print(f"=== chunk {start}..{start + len(chunk)-1}: skip (da xu ly)")
        continue
    print(f"=== chunk {start}..{start + len(chunk)-1} ({len(pending)} mau moi)")
    out = run_gemini_batch(pending)
    for k, v in out["stats"].items():
        agg_stats[k] = agg_stats.get(k, 0) + v
    for row in out["results"]:
        processed_ids.add(row.get("sample_id"))
        all_results.append(row)

    ckpt = OUTPUT_DIR / f"gemini_ckpt_{start:05d}_{start + len(chunk):05d}.json"
    with open(ckpt, "w", encoding="utf-8") as cf:
        json.dump(out, cf, ensure_ascii=False, indent=2, default=str)
    print("checkpoint:", ckpt)

# Sap xep theo thu tu file goc
order = {row.get("sample_id"): i for i, row in enumerate(hasaki_all)}
all_results.sort(key=lambda r: order.get(r.get("sample_id"), 10**9))

BATCH_OUTPUT = {"stats": agg_stats, "results": all_results}
GEMINI_LAST_RUN = {
    "total_in_file": n,
    "labelled": len(all_results),
    "offset": int(HASAKI_RUN_OFFSET),
    "batch_size": bs,
}
print("Xong batch. Stats moi:", agg_stats)
print("Tong ket qua:", len(all_results), "| GEMINI_LAST_RUN:", GEMINI_LAST_RUN)


## 9. Export `hasaki_labelled_gemini_2_5_pro.json`


In [ ]:
if "BATCH_OUTPUT" not in globals() or not BATCH_OUTPUT.get("results"):
    print("Chua co BATCH_OUTPUT — hay chay §8 truoc.")
else:
    export_rows = []
    for row in BATCH_OUTPUT["results"]:
        export_rows.append({
            "sample_id": row.get("sample_id"),
            "product_id": row.get("product_id"),
            "product_name": row.get("product_name"),
            "brand": row.get("brand"),
            "sentence": row.get("sentence"),
            "category": row.get("category"),
            "candidate_intents": row.get("candidate_intents") or [],
            "gemini_intent_3_level": row.get("gemini_intent_3_level") or {},
            "gemini_confidence": row.get("gemini_confidence"),
            "gemini_reasoning": row.get("gemini_reasoning"),
            "gemini_is_new_label": row.get("gemini_is_new_label"),
            "gemini_qa_status": row.get("gemini_qa_status"),
            "gemini_error": row.get("gemini_error"),
        })

    payload = {
        "run_mode": "gemini_2_5_pro_mongodb_labeling",
        "source_input_file": str(HASAKI_JSON),
        "comparison_label_file": str(GPT_LABEL_FILE),
        "model": GEMINI_MODEL,
        "num_labelled": len(export_rows),
        "last_run": GEMINI_LAST_RUN if "GEMINI_LAST_RUN" in globals() else {},
        "stats": BATCH_OUTPUT.get("stats", {}),
        "results": export_rows,
    }

    with open(GEMINI_OUTPUT_FILE, "w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2, default=str)

    print("Da tao:", GEMINI_OUTPUT_FILE)
    print("So ket qua:", len(export_rows))


## 10. So sánh GPT-4o-mini vs Gemini (tùy chọn)

Chạy sau khi đã export Gemini labels. Cần file `hasaki_labelled_clean.json`.


In [ ]:
from collections import Counter

AUDIT_DIR = REPO_ROOT / "data/audit"
AUDIT_DIR.mkdir(parents=True, exist_ok=True)
AGREEMENT_CSV = AUDIT_DIR / "gpt4o_mini_vs_gemini_2_5_pro_label_agreement.csv"
REVIEW_CSV = AUDIT_DIR / "needs_review_gpt4o_mini_vs_gemini_2_5_pro.csv"
METRICS_JSON = AUDIT_DIR / "gemini_2_5_pro_agreement_metrics.json"

if not GPT_LABEL_FILE.exists():
    print("Chua co GPT label file:", GPT_LABEL_FILE)
elif not GEMINI_OUTPUT_FILE.exists():
    print("Chua co Gemini output:", GEMINI_OUTPUT_FILE)
else:
    with open(GPT_LABEL_FILE, "r", encoding="utf-8") as f:
        gpt_data = json.load(f)
    with open(GEMINI_OUTPUT_FILE, "r", encoding="utf-8") as f:
        gem_data = json.load(f)

    gpt_map = {r["sample_id"]: r for r in gpt_data.get("results", [])}
    gem_map = {r["sample_id"]: r for r in gem_data.get("results", [])}
    common_ids = sorted(set(gpt_map) & set(gem_map))

    def agreement_level(g, m):
        g3 = g.get("intent_3_level") or {}
        m3 = m.get("gemini_intent_3_level") or {}
        if m.get("gemini_error") or m.get("gemini_qa_status") == "error":
            return "error"
        if m.get("gemini_is_new_label"):
            return "gemini_new_label"
        l1g, l2g, l3g = g3.get("level_1"), g3.get("level_2"), g3.get("level_3")
        l1m, l2m, l3m = m3.get("level_1"), m3.get("level_2"), m3.get("level_3")
        if l1g == l1m and l2g == l2m and l3g == l3m:
            return "exact_l3_match"
        if l1g == l1m and l2g == l2m:
            return "partial_l2_match"
        if l1g == l1m:
            return "partial_l1_match"
        return "no_match"

    def review_bucket(g, m, level):
        gc = float(g.get("confidence") or 0)
        mc = float(m.get("gemini_confidence") or 0)
        if gc >= 0.90 and mc >= 0.90 and level != "exact_l3_match":
            return "critical_review"
        if level == "exact_l3_match" and gc >= 0.80 and mc >= 0.80:
            return "high_confidence_keep"
        if level == "exact_l3_match":
            return "medium_confidence_keep"
        return "needs_human_review"

    rows = []
    match_l1 = match_l2 = match_l3 = 0
    buckets = Counter()

    for sid in common_ids:
        g, m = gpt_map[sid], gem_map[sid]
        g3, m3 = g.get("intent_3_level") or {}, m.get("gemini_intent_3_level") or {}
        ml1 = g3.get("level_1") == m3.get("level_1")
        ml2 = g3.get("level_2") == m3.get("level_2")
        ml3 = g3.get("level_3") == m3.get("level_3")
        match_l1 += int(ml1)
        match_l2 += int(ml2)
        match_l3 += int(ml3)
        level = agreement_level(g, m)
        bucket = review_bucket(g, m, level)
        buckets[bucket] += 1
        rows.append({
            "sample_id": sid,
            "sentence": g.get("sentence"),
            "category": g.get("category"),
            "gpt_l1": g3.get("level_1"), "gpt_l2": g3.get("level_2"), "gpt_l3": g3.get("level_3"),
            "gpt_confidence": g.get("confidence"), "gpt_qa_status": g.get("qa_status"),
            "gemini_l1": m3.get("level_1"), "gemini_l2": m3.get("level_2"), "gemini_l3": m3.get("level_3"),
            "gemini_confidence": m.get("gemini_confidence"),
            "match_l1": ml1, "match_l2": ml2, "match_l3": ml3,
            "agreement_level": level, "review_bucket": bucket,
        })

    import csv
    with open(AGREEMENT_CSV, "w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()) if rows else [])
        if rows:
            writer.writeheader()
            writer.writerows(rows)

    review_rows = [r for r in rows if r["review_bucket"] in {"needs_human_review", "critical_review"}]
    with open(REVIEW_CSV, "w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(review_rows[0].keys()) if review_rows else [])
        if review_rows:
            writer.writeheader()
            writer.writerows(review_rows)

    n = len(common_ids) or 1
    metrics = {
        "num_samples": len(gpt_map),
        "num_valid_compared": len(common_ids),
        "l1_agreement": round(match_l1 / n, 4),
        "l2_agreement": round(match_l2 / n, 4),
        "l3_agreement": round(match_l3 / n, 4),
        "review_buckets": dict(buckets),
    }
    with open(METRICS_JSON, "w", encoding="utf-8") as f:
        json.dump(metrics, f, ensure_ascii=False, indent=2)

    print("Agreement CSV:", AGREEMENT_CSV)
    print("Review CSV:", REVIEW_CSV)
    print("Metrics:", METRICS_JSON)
    print(json.dumps(metrics, ensure_ascii=False, indent=2))
